# 📝 판다스 데이터 분석 기초 과제 LV3 정답 — 정제 파이프라인 (강사용)

각 단계의 **모범 코드 + 해설** 입니다.

## 1. 쇼핑몰 주문 데이터 정제 파이프라인
**배경**: 방금 내려받은 주문 로그 `shop_orders.csv` 는 결측·이상치·표기 혼재가 섞인 **날것**입니다. 분석 가능한 깨끗한 표로 만들어 저장하는 전 과정을 완성하세요.

아래 각 `### N단계` 셀의 지시대로 **하나의 `df` 를 이어서** 가공합니다. 마지막 자가채점 셀이 최종 `df` 를 검사합니다.

**최종 목표(자가채점 기준)**
| 항목 | 기대값 |
| --- | --- |
| 최종 행 수 | 69 (수량 이상치 1건 제거) |
| `총액` 합계 | 6930000 |
| `시간대` 가 `'저녁'` 인 행 | 27 |
| `총액` 최댓값 | 356000 |

### 1단계 — 데이터 불러오기
`data/shop_orders.csv` 를 `df` 로 불러오세요. 모양은 `(70, 6)` 입니다.

In [ ]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv('../../day06_판다스_기초/data/shop_orders.csv')
print(df.shape)

### 2단계 — 수량 결측 채우기
`수량` 열의 결측을 `1` 로 채우세요 (`fillna(1)`). 채운 뒤 `수량` 결측은 0개가 됩니다.

In [ ]:
df['수량'] = df['수량'].fillna(1)
print(df['수량'].isna().sum())

### 3단계 — 단가 자료형 변환
`단가` 는 `"9,000"` 처럼 콤마 섞인 문자열입니다. 콤마를 지우고 정수형으로 바꿔 다시 `단가` 에 넣으세요.

In [ ]:
df['단가'] = df['단가'].astype(str).str.replace(',', '').astype(int)
print(df['단가'].dtype)

### 4단계 — 문자열 정제
`상품명` 의 앞뒤 공백을 없애고(`.str.strip`), `회원등급` 은 아래 매핑으로 대문자 3종(`GOLD/SILVER/BRONZE`)으로 통일하세요.

```
grade_map = {'gold':'GOLD','GOLD':'GOLD','silver':'SILVER','SILVER':'SILVER','Silver':'SILVER','bronze':'BRONZE','BRONZE':'BRONZE','Bronze':'BRONZE'}
```

In [ ]:
df['상품명'] = df['상품명'].str.strip()
grade_map = {'gold':'GOLD','GOLD':'GOLD','silver':'SILVER','SILVER':'SILVER','Silver':'SILVER','bronze':'BRONZE','BRONZE':'BRONZE','Bronze':'BRONZE'}
df['회원등급'] = df['회원등급'].map(grade_map)
print(df['회원등급'].unique())

### 5단계 — 파생 열 만들기
`총액` = `단가` × `수량` 을 정수형으로 만드세요. 또 `주문일시` 를 날짜형으로 바꿔, 시(hour)가 18 이상이면 `'저녁'`, 아니면 `'주간'` 인 `시간대` 열을 `np.where` 로 만드세요.

In [ ]:
df['총액'] = (df['단가'] * df['수량']).astype(int)
dt = pd.to_datetime(df['주문일시'])
df['시간대'] = np.where(dt.dt.hour >= 18, '저녁', '주간')
print(df[['총액', '시간대']].head())

### 6단계 — 이상치 제거
`수량` 이 100 을 넘는 주문은 오입력 이상치입니다. `수량 <= 100` 인 행만 남겨 다시 `df` 에 넣으세요. (행 수 70 → 69)

In [ ]:
df = df[df['수량'] <= 100]
print(len(df))

### 7단계 — 정제본 저장
`output/` 폴더를 준비하고, 최종 `df` 를 `output/주문_정제본.csv` 로 저장하세요 (`index=False`).

In [ ]:
os.makedirs('output', exist_ok=True)
df.to_csv('output/주문_정제본.csv', index=False)
print('저장 완료:', len(df), '행')

In [ ]:
# [자가채점]
assert df.shape[0] == 69, "수량 이상치 1건 제거 후 69행"
assert str(df['단가'].dtype).startswith('int'), "단가는 정수형"
assert df['수량'].isna().sum() == 0, "수량 결측은 0"
assert set(df['회원등급'].unique()) == {'GOLD', 'SILVER', 'BRONZE'}, "등급 3종으로 통일"
assert df['총액'].sum() == 6930000
assert (df['시간대'] == '저녁').sum() == 27
assert df['총액'].max() == 356000
assert (df['상품명'] == df['상품명'].str.strip()).all(), "상품명 앞뒤 공백이 제거돼야 해요"
print("✅ LV3 문제1 통과!")

### 해설 — 문제 1
- **파이프라인 순서가 핵심**: 결측 채우기 → 형변환 → 문자열 정제 → 파생 → 이상치 제거 → 저장. 앞 단계 결과를 다음 단계가 이어받습니다.
- **왜 이 순서인가**: `총액` 은 `단가`(형변환)와 `수량`(결측 채움)이 정리된 뒤라야 올바르게 곱해집니다. 이상치 제거는 마지막에 해도 `총액` 합계가 6930000 으로 맞아요 (제거된 500개 주문은 어차피 빠지니까요).
- **흔한 실수**: `단가` 를 문자열인 채로 곱하면 문자열이 반복돼(`'9000'*3`) 엉뚱한 값이 됩니다. 곱셈 전에 반드시 정수형으로.
- **참고 통계**: 정제 후 회원등급 분포는 SILVER 25 · BRONZE 24 · GOLD 20 입니다(이상치로 빠진 1건이 SILVER 였습니다).

## 2. 시험 성적 정제·분석
**배경**: `exam_scores.csv` 는 45명의 국·영·수 성적입니다. 결측·이상치·이름 공백을 정리하고, 학생별 총점·평균·등급을 붙여 상·하위를 살펴봅니다. (그룹 집계 없이, 행 단위 계산만 사용)

아래 각 `### N단계` 셀의 지시대로 **하나의 `df` 를 이어서** 가공합니다.

**참고**: 만점은 100점입니다. 100을 넘는 점수는 오입력 이상치예요.

**최종 목표(자가채점 기준)**
| 항목 | 기대값 |
| --- | --- |
| 최종 행 수 | 45 |
| 수학 최댓값 | 100 이하 (이상치 처리 후) |
| 이름 앞뒤 공백 | 없음 |
| `강하은` 의 총점 | 270 (= 99+91+80) |
| 총점 1위 | `강하은` |

### 1단계 — 데이터 불러오기
`data/exam_scores.csv` 를 `df` 로 불러오세요. 모양은 `(45, 7)` 입니다.

In [ ]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv('../../day06_판다스_기초/data/exam_scores.csv')
print(df.shape)

### 2단계 — 이상치 처리
`수학` 이 100 을 넘는 값은 오입력입니다. 100 초과 값을 결측(`np.nan`)으로 바꾸세요. (`df.loc[조건, '수학'] = np.nan`)

In [ ]:
df.loc[df['수학'] > 100, '수학'] = np.nan
print((df['수학'] > 100).sum())

### 3단계 — 결측 채우기
`국어`, `영어`, `수학` 의 결측을 **각 과목 평균의 반올림 정수**로 채우세요. (2단계에서 수학 이상치를 결측 처리했으므로, 평균은 그 뒤 값으로 계산됩니다.)

예: `df['국어'] = df['국어'].fillna(round(df['국어'].mean()))`

In [ ]:
for col in ['국어', '영어', '수학']:
    df[col] = df[col].fillna(round(df[col].mean()))
print(df[['국어', '영어', '수학']].isna().sum().to_dict())

### 4단계 — 이름 공백 정리
`이름` 의 앞뒤 공백을 없애 다시 `이름` 에 넣으세요 (`.str.strip`).

In [ ]:
df['이름'] = df['이름'].str.strip()
print((df['이름'] != df['이름'].str.strip()).sum())

### 5단계 — 행 단위 파생 (총점·평균·등급)
`총점` = 세 과목 합(정수), `평균` = 총점/3 을 소수 2자리로 반올림, `등급` = 평균으로 매기기(90↑ 'A', 80↑ 'B', 70↑ 'C', 그 외 'D'). 등급은 함수를 만들어 `평균` 열에 `apply` 하세요.

In [ ]:
df['총점'] = (df['국어'] + df['영어'] + df['수학']).astype(int)
df['평균'] = (df['총점'] / 3).round(2)

def to_grade(avg):
    if avg >= 90:
        return 'A'
    if avg >= 80:
        return 'B'
    if avg >= 70:
        return 'C'
    return 'D'

df['등급'] = df['평균'].apply(to_grade)
print(df[['이름', '총점', '평균', '등급']].head())

### 6단계 — 상·하위 확인
총점 내림차순으로 정렬해 1위 학생의 `이름` 을 `top_name` 에 담으세요. 또 총점의 최댓값을 확인하세요.

In [ ]:
top_name = df.sort_values('총점', ascending=False).iloc[0]['이름']
print(top_name, df['총점'].max())

### 7단계 — 전체 요약과 저장
`평균` 열의 전체 평균(mean)과 중앙값(median)을 출력하세요. 그리고 `output/` 폴더를 준비해 최종 `df` 를 `output/성적_정제본.csv` 로 저장하세요 (`index=False`).

In [ ]:
print('평균의 평균:', round(df['평균'].mean(), 2), '/ 중앙값:', df['평균'].median())
os.makedirs('output', exist_ok=True)
df.to_csv('output/성적_정제본.csv', index=False)
print('저장 완료:', len(df), '행')

In [ ]:
# [자가채점]
assert df.shape[0] == 45
assert df['수학'].max() <= 100, "이상치 처리 후 수학 최댓값은 100 이하"
assert (df['이름'] != df['이름'].str.strip()).sum() == 0, "이름 앞뒤 공백 없음"
gang = df[df['이름'] == '강하은']
assert len(gang) == 1 and int(gang['총점'].iloc[0]) == 270, "강하은 총점 270"
assert top_name == '강하은', "총점 1위는 강하은"
assert df['총점'].max() == 270
assert round(gang['평균'].iloc[0], 2) == 90.0, "강하은 평균 = 270/3 = 90.0"
assert gang['등급'].iloc[0] == 'A', "평균 90 이상은 A등급"
assert (df['평균'] == (df['총점'] / 3).round(2)).all(), "평균 = 총점/3 (소수 2자리 반올림)"
assert set(df['등급'].unique()) <= {'A', 'B', 'C', 'D'}, "등급은 A/B/C/D 만"
print("✅ LV3 문제2 통과!")

### 해설 — 문제 2
- **파이프라인 순서**: 이상치→결측→공백→파생→분석→저장. 이상치를 먼저 결측으로 바꿔야 그 값이 평균을 오염시키지 않습니다.
- **행 단위 계산**: 총점·평균·등급은 모두 "한 행 안에서" 계산합니다(그룹 집계가 아니에요). 열끼리 더하고, `apply` 로 등급을 매겼습니다.
- **강하은 이 채점 기준인 이유**: 강하은은 세 과목 모두 값이 있고(99·91·80) 이상치도 없어, 결측 채우기와 **무관하게** 총점이 항상 270 으로 확정됩니다. 채점을 안정적으로 만드는 기준점이에요.
- **참고 통계**: 결측을 반올림 평균(국어 71·영어 68·수학 69)으로 채우면 평균의 평균은 약 69.19, 중앙값은 70.0, 등급 분포는 A 1 · B 9 · C 13 · D 22 명입니다.
- **흔한 실수**: 반올림에 `round()` 를 쓰면 파이썬은 0.5 를 짝수로 올림/내림합니다. 이 데이터의 과목 평균들은 .5 경계가 아니라 결과가 71/68/69 로 일정합니다.